In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install transformers rank-bm25 faiss-cpu sentence-transformers -q

import torch
import torch.nn as nn
import json
import numpy as np
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import faiss
from tqdm import tqdm

DRIVE_BASE = "/content/drive/MyDrive/medrag"
DATA_PATH = f"{DRIVE_BASE}/pubmedqa_filtered.json"
OUTPUT_DIR = f"{DRIVE_BASE}/trial1_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(DATA_PATH, "r") as f:
    corpus = json.load(f)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Corpus loaded: {len(corpus)} samples")
print(f"Output dir: {OUTPUT_DIR}")

In [ ]:
MODEL_ID = "stanford-crfm/BioMedLM"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    attn_implementation="eager"
)
model = model.to("cuda")
model.eval()

for param in model.parameters():
    param.requires_grad = False

n_layers = model.config.n_layer
hidden_size = model.config.n_embd
n_heads = model.config.n_head

print(f"Model loaded: {n_layers} layers, {n_heads} heads, hidden size {hidden_size}")

In [ ]:
documents = []
doc_metadata = []

for sample in corpus:
    for abstract in sample["supporting_abstracts"]:
        documents.append(abstract)
        doc_metadata.append({
            "pubid": sample["pubid"],
            "query": sample["query"],
            "gold_answer": sample["gold_answer"],
            "label": sample["label"]
        })

tokenized_docs = [doc.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_docs)

enc_model = SentenceTransformer("NeuML/pubmedbert-base-embeddings")
doc_embeddings = enc_model.encode(documents, batch_size=64,
                                   show_progress_bar=True,
                                   convert_to_numpy=True)
faiss.normalize_L2(doc_embeddings)
index = faiss.IndexFlatIP(doc_embeddings.shape[1])
index.add(doc_embeddings)

print(f"Retriever ready: {index.ntotal} documents indexed")

In [ ]:
FAISS_RELEVANCE_THRESHOLD = 0.5

def bm25_retrieve(query, k=3):
    scores = bm25.get_scores(query.lower().split())
    top_k = np.argsort(scores)[::-1][:k]
    return [{"abstract": documents[i], "pubid": doc_metadata[i]["pubid"],
             "score": float(scores[i]), "gold_answer": doc_metadata[i]["gold_answer"],
             "label": doc_metadata[i]["label"]} for i in top_k]

def faiss_retrieve(query, k=3):
    qe = enc_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(qe)
    scores, indices = index.search(qe, k)
    return [{"abstract": documents[i], "pubid": doc_metadata[i]["pubid"],
             "score": float(s), "gold_answer": doc_metadata[i]["gold_answer"],
             "label": doc_metadata[i]["label"]} for s, i in zip(scores[0], indices[0])]

def hybrid_retrieve(query, k=3, rrf_k=60):
    bm25_results = bm25_retrieve(query, k=k*2)
    faiss_results = faiss_retrieve(query, k=k*2)
    combined = {}
    for rank, r in enumerate(bm25_results):
        combined[r["abstract"]] = {"meta": r, "score": 1 / (rrf_k + rank + 1),
                                    "faiss_score": 0.0}
    for rank, r in enumerate(faiss_results):
        key = r["abstract"]
        if key in combined:
            combined[key]["score"] += 1 / (rrf_k + rank + 1)
            combined[key]["faiss_score"] = r["score"]
        else:
            combined[key] = {"meta": r, "score": 1 / (rrf_k + rank + 1),
                             "faiss_score": r["score"]}
    sorted_results = sorted(combined.values(),
                           key=lambda x: x["score"], reverse=True)[:k]
    for item in sorted_results:
        item["meta"]["faiss_score"] = item["faiss_score"]
    return [item["meta"] for item in sorted_results]

def build_rag_prompt_with_spans(query, k=3):
    results = hybrid_retrieve(query, k=k)
    context_parts = [f"[{i+1}] {r['abstract']}" for i, r in enumerate(results)]
    context = "\n\n".join(context_parts)
    prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"

    span_index = []
    prefix_tokens = tokenizer("Context:\n", return_tensors="pt")["input_ids"].shape[1]
    tokens_so_far = prefix_tokens

    for i, part in enumerate(context_parts):
        part_len = tokenizer(part, return_tensors="pt")["input_ids"].shape[1]
        sep_len = tokenizer("\n\n", return_tensors="pt")["input_ids"].shape[1]
        span_index.append({
            "doc_rank": i,
            "pubid": results[i]["pubid"],
            "token_start": tokens_so_far,
            "token_end": tokens_so_far + part_len,
            "gold_answer": results[i]["gold_answer"],
            "label": results[i]["label"],
            "faiss_score": results[i].get("faiss_score", 0.0)
        })
        tokens_so_far += part_len + sep_len

    has_distractor = any(s["faiss_score"] < FAISS_RELEVANCE_THRESHOLD
                        for s in span_index)

    return prompt, results, span_index, has_distractor

print("Retrieval and prompt functions ready.")

In [ ]:
QUESTION_MARKER = [198, 198, 48, 1192, 281, 25]
LATE_LAYERS = [28, 29, 30, 31]

def find_question_start(input_ids_list):
    for i in range(len(input_ids_list) - len(QUESTION_MARKER)):
        if input_ids_list[i:i+len(QUESTION_MARKER)] == QUESTION_MARKER:
            return i + len(QUESTION_MARKER)
    return None

def compute_head_entropy(attn_weights):
    """
    Shannon entropy of attention distribution per head.
    attn_weights: [heads, seq_len] — attention from one position to all keys
    Returns: [heads] entropy values
    """
    p = attn_weights + 1e-10
    return -(p * torch.log(p)).sum(dim=-1)

def compute_evidence_to_evidence(attn_squeezed, span_index, seq_len):
    """
    Mean attention from evidence tokens to other evidence tokens.
    Measures coherence — are the abstracts being synthesized together?
    attn_squeezed: [heads, seq_len, seq_len]
    """
    evidence_positions = []
    for span in span_index:
        start = span["token_start"]
        end = min(span["token_end"], seq_len)
        if start < seq_len:
            evidence_positions.extend(range(start, end))

    if len(evidence_positions) < 2:
        return 0.0

    ev_pos = torch.tensor(evidence_positions, dtype=torch.long)
    attn_ev_to_ev = attn_squeezed[:, ev_pos[:, None], ev_pos[None, :]]
    return float(attn_ev_to_ev.mean())

print("Helper functions ready.")

In [ ]:
N_SAMPLES = len(corpus)

trial1_results = []
skipped = 0

print(f"Running Trial 1 on {N_SAMPLES} samples...\n")

for idx, sample in enumerate(tqdm(corpus[:N_SAMPLES])):
    query = sample["query"]

    rag_prompt, retrieved, span_index, has_distractor = \
        build_rag_prompt_with_spans(query, k=3)

    inputs = tokenizer(rag_prompt, return_tensors="pt",
                      truncation=True, max_length=512).to("cuda")
    seq_len = inputs["input_ids"].shape[1]

    input_ids_list = inputs["input_ids"][0].tolist()
    question_start = find_question_start(input_ids_list)

    if question_start is None or question_start >= seq_len:
        question_start = max(0, seq_len - 10)
        truncated = True
    else:
        truncated = question_start > 400

    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True)

    attention_trajectory = []
    sink_trajectory = []
    ev_to_ev_trajectory = []
    late_layer_head_data = {}

    for layer_idx, attn in enumerate(outputs.attentions):
        attn_sq = attn.squeeze(0)

        sink_attn = attn_sq[:, question_start:, 0].mean().item()
        sink_trajectory.append(float(sink_attn))

        if layer_idx == 0:
            attention_trajectory.append(0.0)
            ev_to_ev_trajectory.append(0.0)
            continue

        ev_ev = compute_evidence_to_evidence(attn_sq, span_index, seq_len)
        ev_to_ev_trajectory.append(float(ev_ev))

        layer_evidence_mass = 0.0
        n_spans = 0

        for span in span_index:
            start = span["token_start"]
            end = min(span["token_end"], seq_len)
            if start >= seq_len or end <= start:
                continue

            attn_q_to_span = attn_sq[:, question_start:, start:end]
            per_head = attn_q_to_span.mean(dim=[1, 2])
            layer_evidence_mass += per_head.max().item()
            n_spans += 1

        if n_spans > 0:
            layer_evidence_mass /= n_spans
        attention_trajectory.append(float(layer_evidence_mass))

        if layer_idx in LATE_LAYERS:
            head_attentions = []
            head_entropies = []

            for h in range(n_heads):
                head_attn_full = attn_sq[h, question_start:, :]
                mean_head_attn = head_attn_full.mean(dim=0)

                ev_attn = 0.0
                n_sp = 0
                for span in span_index:
                    start = span["token_start"]
                    end = min(span["token_end"], seq_len)
                    if start < seq_len and end > start:
                        ev_attn += mean_head_attn[start:end].mean().item()
                        n_sp += 1
                if n_sp > 0:
                    ev_attn /= n_sp
                head_attentions.append(float(ev_attn))

                entropy = compute_head_entropy(
                    head_attn_full.mean(dim=0).unsqueeze(0)
                )
                head_entropies.append(float(entropy[0]))

            late_layer_head_data[layer_idx] = {
                "head_attentions": head_attentions,
                "head_entropies": head_entropies,
                "top_head": int(np.argmax(head_attentions)),
                "top_head_attention": float(max(head_attentions)),
                "top_head_entropy": float(head_entropies[int(np.argmax(head_attentions))])
            }

    trial1_results.append({
        "idx": idx,
        "pubid": sample["pubid"],
        "query": query[:60],
        "label": sample["label"],
        "seq_len": seq_len,
        "question_start": question_start,
        "truncated": truncated,
        "has_distractor": has_distractor,
        "attention_trajectory": attention_trajectory,
        "sink_trajectory": sink_trajectory,
        "ev_to_ev_trajectory": ev_to_ev_trajectory,
        "mean_attention": float(np.mean(attention_trajectory[1:])),
        "max_attention_layer": int(np.argmax(attention_trajectory[1:]) + 1),
        "max_attention_value": float(np.max(attention_trajectory[1:])),
        "final_layer_attention": attention_trajectory[-1],
        "mean_sink": float(np.mean(sink_trajectory)),
        "mean_ev_to_ev": float(np.mean(ev_to_ev_trajectory[1:])),
        "late_layer_heads": late_layer_head_data
    })

    if (idx + 1) % 50 == 0:
        checkpoint_path = os.path.join(OUTPUT_DIR, f"trial1_checkpoint_{idx+1}.json")
        with open(checkpoint_path, "w") as f:
            json.dump(trial1_results, f)
        tqdm.write(f"Checkpoint saved at sample {idx+1}")

print(f"\nTrial 1 complete. {len(trial1_results)} samples processed.")
print(f"\nSummary:")
print(f"  Mean attention to evidence: "
      f"{np.mean([r['mean_attention'] for r in trial1_results]):.6f}")
print(f"  Mean peak layer: "
      f"{np.mean([r['max_attention_layer'] for r in trial1_results]):.1f}")
print(f"  Mean attention sink: "
      f"{np.mean([r['mean_sink'] for r in trial1_results]):.6f}")
print(f"  Truncated samples: "
      f"{sum(1 for r in trial1_results if r['truncated'])}")
print(f"  Distractor samples: "
      f"{sum(1 for r in trial1_results if r['has_distractor'])}")

In [ ]:
output_path = os.path.join(OUTPUT_DIR, "trial1_full_results.json")
with open(output_path, "w") as f:
    json.dump({
        "n_samples": len(trial1_results),
        "n_layers": n_layers,
        "n_heads": n_heads,
        "late_layers_analyzed": LATE_LAYERS,
        "summary": {
            "mean_attention": float(np.mean([r["mean_attention"] for r in trial1_results])),
            "mean_peak_layer": float(np.mean([r["max_attention_layer"] for r in trial1_results])),
            "mean_sink": float(np.mean([r["mean_sink"] for r in trial1_results])),
            "mean_ev_to_ev": float(np.mean([r["mean_ev_to_ev"] for r in trial1_results])),
            "truncated_count": sum(1 for r in trial1_results if r["truncated"]),
            "distractor_count": sum(1 for r in trial1_results if r["has_distractor"]),
            "label_breakdown": {
                "yes": sum(1 for r in trial1_results if r["label"] == "yes"),
                "no": sum(1 for r in trial1_results if r["label"] == "no"),
                "maybe": sum(1 for r in trial1_results if r["label"] == "maybe")
            }
        },
        "samples": trial1_results
    }, f, indent=2)

for f in os.listdir(OUTPUT_DIR):
    if f.startswith("trial1_checkpoint"):
        os.remove(os.path.join(OUTPUT_DIR, f))

print(f"Full results saved to {output_path}")
print(f"Trial 1: COMPLETE ✓")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import json
import numpy as np

with open("/content/drive/MyDrive/medrag/trial1_outputs/trial1_full_results.json") as f:
    results = json.load(f)

print(f"Samples loaded: {results['n_samples']}")
print(f"Summary: {results['summary']}")

distractor_count = results['summary']['distractor_count']
total = results['n_samples']
print(f"\nDistractor rate: {distractor_count}/{total} = {distractor_count/total:.1%}")

sinks = [r['mean_sink'] for r in results['samples']]
attentions = [r['mean_attention'] for r in results['samples']]
print(f"\nSink stats: mean={np.mean(sinks):.3f} min={min(sinks):.3f} max={max(sinks):.3f}")
print(f"Evidence attn stats: mean={np.mean(attentions):.6f} min={min(attentions):.6f} max={max(attentions):.6f}")

In [ ]:
import json
import numpy as np

with open("/content/drive/MyDrive/medrag/trial1_outputs/trial1_full_results.json") as f:
    results = json.load(f)

faiss_scores = []
for r in results["samples"]:
    for span in r.get("late_layer_heads", {}).values():
        pass

distractor_samples = [r for r in results["samples"] if r["has_distractor"]]
clean_samples = [r for r in results["samples"] if not r["has_distractor"]]

print(f"Distractor samples: {len(distractor_samples)}")
print(f"Clean samples: {len(clean_samples)}")

print(f"\nDistractor mean attention: {np.mean([r['mean_attention'] for r in distractor_samples]):.6f}")
print(f"Clean mean attention:      {np.mean([r['mean_attention'] for r in clean_samples]):.6f}")

print(f"\nDistractor mean sink: {np.mean([r['mean_sink'] for r in distractor_samples]):.4f}")
print(f"Clean mean sink:      {np.mean([r['mean_sink'] for r in clean_samples]):.4f}")

print(f"\nBy label:")
for label in ["yes", "no"]:
    label_samples = [r for r in results["samples"] if r["label"] == label]
    print(f"  {label}: mean_attn={np.mean([r['mean_attention'] for r in label_samples]):.6f} "
          f"mean_sink={np.mean([r['mean_sink'] for r in label_samples]):.4f} "
          f"n={len(label_samples)}")

In [ ]:
from google.colab import runtime
runtime.unassign()